# 04 - S3.3b: T4 verification, epoch 0, smoke run, throughput benchmark (Stage 3)
Implements `docs/s33b_spec.md`. **Nothing here is a thesis result.** Runtime: **T4 GPU**. Run the cells top to bottom.

**Session 1 (about 20-25 min):** set `SESSION = 1` below. Verifies the platform (chat template, equality with S3.0,
fresh KV cache, Colab tests). At the end a results file downloads: **upload it to the chat** and wait until I say the
session-2 code is pushed.

**Session 2 (about 2 h, three runtimes):** set `SESSION = 2`. Registers platform epoch 0, runs the smoke units, and
**kills the runtime twice on purpose** to test resume and Drive. When the runtime disappears:
1. Wait about 10 seconds, then **Runtime -> Disconnect and delete runtime** (if it is not already gone), then
   **Connect** (T4 GPU again; if no T4 is offered, wait and retry - never continue on CPU).
2. Run the cells again from the top. Each cell knows what is already done and continues where it stopped.

After the second kill and the benchmark, a results file downloads: **upload it to the chat**.

Never edit files inside `/content/eem-acc-skripsi` (a changed checkout stops the run: it must stay clean).

In [ ]:
# 0) Settings
SESSION = 1          # 1 = verification; 2 = epoch 0 + smoke + kills + benchmark (only after I push the session-2 code)
SESSION_TAG = {1: "s33b-session1", 2: "s33b-session2"}[SESSION]

In [ ]:
# 1) Setup: timeline, GPU, Drive, pinned code (every runtime)
import os, sys, subprocess, tempfile, json, time, glob, shutil, hashlib
from pathlib import Path
T_FIRST_CELL = time.time()
REPO_URL = "https://github.com/notanut/eem-acc-skripsi.git"
IN_COLAB = "google.colab" in sys.modules
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
print("GPU:", gpu.stdout.strip() or "NONE")
if IN_COLAB and "T4" not in gpu.stdout:
    raise RuntimeError("No T4 GPU: Runtime -> Change runtime type -> T4 GPU (if none is offered, wait and retry).")
if IN_COLAB:
    from google.colab import drive
    t0 = time.time(); drive.mount("/content/drive"); T_MOUNT = time.time() - t0
    REPO_DIR = "/content/eem-acc-skripsi"
    DRIVE_ROOT = "/content/drive/MyDrive/Skripsi"
    def git(*args, cwd=REPO_DIR, capture=False):
        return subprocess.run(["git", *args], cwd=cwd, check=True, capture_output=capture, text=True)
    if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
        git("clone", REPO_URL, REPO_DIR, cwd=None)
    git("fetch", "--tags", "--force", "origin")
    target = git("rev-parse", f"{SESSION_TAG}^{{commit}}", capture=True).stdout.strip()
    head = git("rev-parse", "HEAD", capture=True).stdout.strip()
    if head != target:
        if "eem_acc" in sys.modules:
            raise RuntimeError("Other code is loaded: Runtime -> Restart session, then run from the top.")
        git("checkout", "--detach", target)
    dirty = git("status", "--porcelain", "--untracked-files=no", capture=True).stdout.strip()
    assert not dirty, f"the checkout is not clean: {dirty}"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements.txt", "-e", REPO_DIR], check=True)
else:                                   # local test run: this checkout + a temp folder
    T_MOUNT = 0.0
    REPO_DIR = str(Path.cwd().parent) if Path.cwd().name == "notebooks" else str(Path.cwd())
    DRIVE_ROOT = tempfile.mkdtemp(prefix="eem_acc_")
import importlib
if f"{REPO_DIR}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/src")
importlib.invalidate_caches()
PINS_FILE = Path(REPO_DIR) / "configs" / "t4_pins.json"
PINS = json.loads(PINS_FILE.read_text()) if PINS_FILE.exists() else {}
if SESSION == 2 and IN_COLAB:          # install the pinned libraries BEFORE anything imports transformers
    if "transformers" in sys.modules:
        raise RuntimeError("transformers is already imported: Runtime -> Restart session, then run from the top.")
    want = [f"{k.replace('_', '-')}=={v}" for k, v in PINS.get("libraries", {}).items()]
    if want:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *want], check=True)
from eem_acc.stage3 import feasibility as F
from eem_acc.stage3.llm import t4, runner as R
from eem_acc.stage3.llm.ledger import vm_boot_time
ROOT = Path(DRIVE_ROOT) / "stage3"                     # the campaign root (runner, epochs, ledger, lease, claims)
OUT = ROOT / "s33b"; OUT.mkdir(parents=True, exist_ok=True)
REPO_COMMIT = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, capture_output=True, text=True).stdout.strip()
S = t4.Sections(OUT / f"session{SESSION}.json", stamp={"commit": REPO_COMMIT, "tag": SESSION_TAG})
RUNTIME_INFO = {"vm_boot": vm_boot_time(), "first_cell": T_FIRST_CELL, "mount_seconds": T_MOUNT, "commit": REPO_COMMIT,
                "gpu": gpu.stdout.strip()}
if S.data.get("stamp", {}).get("commit") not in (None, REPO_COMMIT):
    raise RuntimeError(f"this session was started at another commit ({S.data['stamp']['commit']}); ask in the chat")
print("commit", REPO_COMMIT[:12], "| saved sections:", [k for k in S.data if k != "stamp"])

In [ ]:
# 2) Install llama-cpp-python with CUDA (pinned; the built wheel is cached on Drive)
import zipfile, zlib
LLAMA_VERSION = "0.3.36"           # released 2026-10-01 (PyPI); pinned for reproducibility
PYTAG = f"cp{sys.version_info.major}{sys.version_info.minor}"
WHEEL_DIR = Path(DRIVE_ROOT) / "wheels"
WHEEL_DIR.mkdir(parents=True, exist_ok=True)
BUILD_DIR = Path(globals().get("BUILD_DIR_OVERRIDE", "/content/wheels"))   # override only in tests
def compatible(name):
    """llama-cpp-python loads its library through ctypes, so a source build is tagged py3-none-linux_x86_64;
    prebuilt wheels may carry the CPython tag instead. Accept both, never another Python version."""
    parts = name[:-4].split("-")          # dist, version, python tag, abi tag, platform
    return len(parts) == 5 and parts[2] in (PYTAG, "py3") and parts[4].startswith("linux_x86_64")
def good_wheels(folder=None):
    out = []
    for w in sorted(Path(folder or WHEEL_DIR).glob(f"llama_cpp_python-{LLAMA_VERSION}-*.whl")):
        if not compatible(w.name):
            continue
        try:
            with zipfile.ZipFile(w) as z:
                bad = z.testzip()
            if bad is None:
                out.append(w)
            else:
                print(f"ignoring a damaged wheel {w.name} (bad member {bad})")
        except (zipfile.BadZipFile, OSError, EOFError, zlib.error) as e:
            print(f"ignoring a broken wheel {w.name}: {e}")
    return out
def installed_version():
    r = subprocess.run([sys.executable, "-c", "import llama_cpp; print(llama_cpp.__version__)"], capture_output=True, text=True)
    return (r.stdout or "").strip() if r.returncode == 0 else None
def import_error():
    r = subprocess.run([sys.executable, "-c", "import llama_cpp"], capture_output=True, text=True)
    return (r.stderr or "").strip().splitlines()[-1:] if r.returncode != 0 else []
def save_to_drive(wheels):
    for w in wheels:
        tmp = WHEEL_DIR / (w.name + ".tmp")
        shutil.copy(w, tmp)
        os.replace(tmp, WHEEL_DIR / w.name)                 # never leave a half-copied wheel under the real name
    saved = good_wheels()
    if not saved:
        raise RuntimeError(f"copying the wheel to Drive failed; it is still in {BUILD_DIR}")
    return saved
already = installed_version() == LLAMA_VERSION
cached = good_wheels()
if IN_COLAB and not cached and good_wheels(BUILD_DIR):
    cached = save_to_drive(good_wheels(BUILD_DIR))          # an earlier build in this session that was not saved
    print(f"saved an earlier build from {BUILD_DIR} to Drive")
if IN_COLAB and not cached and not already:
    t0 = time.time()
    print("Building llama-cpp-python with CUDA for the T4 (sm_75). This takes about an hour the first time ...")
    env = dict(os.environ, CMAKE_ARGS="-DGGML_CUDA=on -DCMAKE_CUDA_ARCHITECTURES=75", CMAKE_BUILD_PARALLEL_LEVEL="4")
    subprocess.run([sys.executable, "-m", "pip", "wheel", "--no-deps", f"llama-cpp-python=={LLAMA_VERSION}",
                    "-w", str(BUILD_DIR)], env=env, check=True)
    built = good_wheels(BUILD_DIR)
    if not built:
        raise RuntimeError(f"build finished but no usable wheel in {BUILD_DIR}: {os.listdir(BUILD_DIR)}")
    cached = save_to_drive(built)
    print(f"built in {(time.time()-t0)/60:.1f} min")
print("cached on Drive:", [w.name for w in cached])
if cached and not already:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", str(cached[-1])], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy", "typing-extensions", "diskcache", "jinja2"], check=True)
if already and not cached:
    print("WARNING: llama-cpp-python is installed but no wheel is cached on Drive; wheel sha256 unknown")
# sha256 of the cached wheel (if the library was installed from elsewhere earlier in this session, it may differ)
WHEEL_SHA256 = hashlib.sha256(Path(cached[-1]).read_bytes()).hexdigest() if cached else None
print("llama-cpp-python:", installed_version(), "| wheel sha256:", WHEEL_SHA256)
if IN_COLAB and installed_version() != LLAMA_VERSION:
    hint = f" (delete {cached[-1]} on Drive to force a rebuild)" if cached else ""
    raise RuntimeError(f"llama-cpp-python {LLAMA_VERSION} is not importable: {import_error()}{hint}")

In [ ]:
# 3) Model: Drive cache (S3.0), local copy verified against S3.0's sha256 (every runtime)
from huggingface_hub import list_repo_files, hf_hub_download
MODEL_REPO, QUANT = "Qwen/Qwen2.5-7B-Instruct-GGUF", "q4_k_m"
MODEL_DIR = Path(DRIVE_ROOT) / "models" / MODEL_REPO.replace("/", "__")
files = sorted(f for f in list_repo_files(MODEL_REPO) if f.lower().endswith(".gguf") and QUANT in f.lower())
LOCAL = Path("/content/models") if IN_COLAB else Path(tempfile.mkdtemp())
LOCAL.mkdir(parents=True, exist_ok=True)
for f in files:
    src = Path(hf_hub_download(MODEL_REPO, f, local_dir=MODEL_DIR))
    dst = LOCAL / f
    if not dst.exists() or dst.stat().st_size != src.stat().st_size:
        shutil.copy(src, str(dst) + ".tmp"); os.replace(str(dst) + ".tmp", dst)
MODEL_PATHS = [LOCAL / f for f in files]
PARTS = t4.verify_parts(MODEL_PATHS)
assert PARTS["equal_s30"], f"the model differs from S3.0's: {PARTS}"
PART_SHA = PARTS["sha256"]
print("model verified:", list(PART_SHA))

In [ ]:
# 4) Load the model on the GPU (every runtime)
import gc
from llama_cpp import Llama
for name in ("llm",):
    if globals().get(name) is not None:
        globals()[name].close(); globals()[name] = None
gc.collect()
t0 = time.time()
llm = Llama(model_path=str(MODEL_PATHS[0]), n_gpu_layers=-1, n_ctx=8192, n_batch=512, seed=0, verbose=False)
LOAD_SECONDS = time.time() - t0
print(f"loaded in {LOAD_SECONDS:.0f} s;", t4.gpu_telemetry())

In [ ]:
# 5) SESSION 1: verification (template, S3.0 equality, load params, libraries, canary counts, versions, timing)
if SESSION == 1:
    from eem_acc.stage3.llm.llama_backend import LlamaCppBackend, LlamaTokenizer
    from eem_acc.stage3.llm.canary import load_reference_prompts
    s30 = json.loads((Path(REPO_DIR) / "docs/evidence/s30_results_2026-11-21.json").read_text())
    B = LlamaCppBackend(llm, t4.model_digest(PART_SHA), WHEEL_SHA256)
    TOKZ = LlamaTokenizer(B)
    S.save("runtime", RUNTIME_INFO)
    if not S.done("template"):
        S.save("template", t4.template_check(llm))
    if not S.done("s30_equality"):
        chat = lambda m, g: (llm.reset(), F.make_generate(llm, grammar=g, reset=True, max_tokens=t4.S30_MAX_TOKENS)(m))[1]
        S.save("s30_equality", t4.s30_equality(B, s30["timing"]["records"], F.sample_prompts(20, seed=0), chat))
    if not S.done("load_params"):
        S.save("load_params", {**t4.load_params_report(llm, s30["load"]["stamp"]), "cuda_libraries": t4.cuda_libraries()})
    if not S.done("canary_counts"):
        S.save("canary_counts", t4.canary_counts(TOKZ, load_reference_prompts()[0]))
    if not S.done("versions"):
        S.save("versions", {"libraries": t4.library_versions(), "minilm_revision": t4.minilm_revision()})
    if not S.done("timing"):
        S.save("timing", t4.timing_check(S.data["s30_equality"]["records"]))
    eq = S.data["s30_equality"]
    print("template equals S3.0:", S.data["template"]["equals_s30"], "| S3.0 equality:", eq["equal"], "/", eq["compared"])
    print("timing:", S.data["timing"], "| canary counts equal:", S.data["canary_counts"]["equal"])
    if not (S.data["template"]["equals_s30"] and eq["all_equal"]):
        print(">>> STOP: the platform check failed. Upload the results file (last cell) to the chat. <<<")

In [ ]:
# 6) SESSION 1: D47 on the GPU (reset path vs a freshly constructed model; at most two models at a time)
if SESSION == 1 and not S.done("d47"):
    longest = sorted(load_reference_prompts()[0], key=lambda p: -p["n_tokens"])[:2]
    fresh = lambda: LlamaCppBackend(Llama(model_path=str(MODEL_PATHS[0]), n_gpu_layers=-1, n_ctx=8192, n_batch=512,
                                          seed=0, verbose=False), t4.model_digest(PART_SHA), WHEEL_SHA256)
    S.save("d47", t4.d47_check(B, fresh, longest, TOKZ))
    gc.collect()
if SESSION == 1:
    print("D47:", S.data["d47"])

In [ ]:
# 7) SESSION 1: Colab-marked tests (their own process loads one more model: two in total)
if SESSION == 1 and not S.done("colab_tests"):
    env = dict(os.environ, EEM_GGUF=str(MODEL_PATHS[0]), EEM_MINILM_REV=S.data["versions"]["minilm_revision"] or "",
               PYTHONPATH=f"{REPO_DIR}/src:{REPO_DIR}/tests")
    r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-m", "colab", "-p", "no:cacheprovider",
                        "tests/test_s33_colab.py"], cwd=REPO_DIR, env=env, capture_output=True, text=True)
    S.save("colab_tests", {"returncode": r.returncode, "tail": (r.stdout + r.stderr)[-4000:]})
if SESSION == 1:
    print(S.data["colab_tests"]["tail"][-800:])

In [ ]:
# 8) SESSION 2: open the runtime (canary -> epoch; lease; claims of dead runtimes broken) - every runtime
if SESSION == 2:
    from eem_acc.stage3.llm.embed import MiniLMEmbedder
    EMB = MiniLMEmbedder(PINS["minilm_revision"])
    SES = t4.T4Session(ROOT, llm, PART_SHA, WHEEL_SHA256, EMB, session_id="s33b-2", load_seconds=LOAD_SECONDS)
    RUN = SES.runner
    S.append("runtimes", {**RUNTIME_INFO, "runtime": SES.runtime, "epoch": SES.epoch, "canary": SES.status,
                          "fingerprint": SES.fingerprint, "broken_claims": SES.broken_claims,
                          "canary_seconds": SES.canary_seconds, "load_seconds": LOAD_SECONDS})
    print(f"runtime {len(S.data['runtimes'])}: epoch {SES.epoch} ({SES.status})")
    if len(S.data["runtimes"]) > 1 and SES.status != "pass":
        print(">>> the canary did not pass in this runtime: the resume test is inconclusive (it is recorded). <<<")
    REF, (U_C2, U_C0, U_RQ3) = t4.smoke_plan()
    for phase, units in (("smoke-ref", REF), ("smoke", [U_C2, U_C0, U_RQ3])):
        if not (ROOT / "plans" / f"{phase}.json").exists():
            R.write_plan(ROOT, phase, units, 1)

In [ ]:
# 9) SESSION 2: smoke run with the two forced kills (re-run this cell after each reconnect)
if SESSION == 2:
    RUN.run_phase("smoke-ref")                                   # the uninterrupted reference, first
    if RUN.cache_path(U_C2).exists() and not RUN.is_done(U_C2) and not S.done("kill_a_drive"):
        S.save("kill_a_drive", {**t4.drive_state(RUN.cache_path(U_C2)),        # read BEFORE the cache is opened
                                "records_expected": t4.records_at_kill(RUN.cache_path(REF[0]), t4.KILL_A_AFTER)})
    if RUN.result_path(U_C0).exists() and not S.done("kill_b_drive"):
        S.save("kill_b_drive", {**t4.drive_state(RUN.cache_path(U_C0), RUN.result_path(U_C0)),
                                "is_done": RUN.is_done(U_C0, full=True)})
    if t4.arm_kill_a(RUN, U_C2):
        print(f">>> KILL (a): the runtime will be killed after {t4.KILL_A_AFTER} LLM calls of {U_C2.unit_id}. <<<")
        RUN.ctx.backend = t4.KillAfter(SES.backend, t4.KILL_A_AFTER, t4.colab_kill)
    else:
        RUN.ctx.backend = SES.backend
        if t4.arm_kill_b(RUN, U_C0, t4.colab_kill):
            print(f">>> KILL (b): the runtime will be killed right after {U_C0.unit_id}'s result is written. <<<")
        else:
            RUN.after_result = None
    out = RUN.run_phase("smoke")
    S.save("smoke_run", out)
    res_c2 = R.split_result(json.loads(RUN.result_path(U_C2).read_text()))[1]
    if not S.done("kill_a_resumed"):
        S.save("kill_a_resumed", {"resumed": res_c2["resumed"], "restarts": res_c2["restarts"],
                                  "cross_cpu_resume": res_c2.get("cross_cpu_resume")})
    print("smoke:", out, "| kill (a):", S.data.get("kill_a_drive"), "| kill (b):", S.data.get("kill_b_drive"))

In [ ]:
# 10) SESSION 2: smoke checks (interrupted vs reference, verify_replay on CPU, audit, statistics, Drive latency)
if SESSION == 2 and not S.done("smoke_checks"):
    from eem_acc.stage3.llm.audit import audit_phase
    from eem_acc.stage3.llm.llama_backend import key_source_from_header, LlamaTokenizer
    from eem_acc.stage3.llm.cache import read_header
    load = lambda u: R.split_result(json.loads(RUN.result_path(u).read_text()))[0]
    cmp = t4.compare_units(RUN.cache_path(U_C2), RUN.cache_path(REF[0]), load(U_C2), load(REF[0]))
    vocab = Llama(model_path=str(MODEL_PATHS[0]), vocab_only=True, verbose=False)
    verified = {}
    for u in REF + [U_C2, U_C0, U_RQ3]:
        ks = key_source_from_header(read_header(RUN.cache_path(u)), vocab)
        try:
            verified[u.unit_id] = R.verify_replay(ROOT, u.phase, u.unit_id, ks, LlamaTokenizer(ks))
        except Exception as e:
            verified[u.unit_id] = f"{type(e).__name__}: {e}"
    results = [load(u) for u in REF + [U_C2, U_C0, U_RQ3]]
    S.save("smoke_checks", {"interrupted_vs_reference": cmp, "verify_replay": verified,
                            "audit": {k: v for k, v in audit_phase(ROOT, "smoke").items() if k != "hits"} | {
                                "hits": audit_phase(ROOT, "smoke")["hits"][:20]},
                            "stats": t4.smoke_stats(results, lambda uj: RUN.cache_path(R.Unit.from_json(uj))),
                            "append_latency": t4.append_latency(ROOT / "s33b" / "latency.jsonl")})
    heavy = ("steps", "note_events", "llm_calls", "final_notes", "embedding_cache", "window_trace")
    S.save("smoke_results", [{k: v for k, v in r.items() if k not in heavy} for r in results])   # slim; caches on Drive
if SESSION == 2:
    print(json.dumps({k: S.data["smoke_checks"][k] for k in ("interrupted_vs_reference", "verify_replay")}, indent=1))

In [ ]:
# 11) SESSION 2: D53 benchmark A (fresh) vs B (static-prefix snapshot) - the GPU-hour counter starts here
if SESSION == 2 and not S.done("bench_ab"):
    from eem_acc.stage3.llm.llama_backend import LlamaCppBackend
    from eem_acc.stage3.llm.canary import run_canary, fingerprint
    if not S.done("benchmark_start"):
        SES.ledger.event("benchmark_start"); S.save("benchmark_start", time.time())
    caches = [RUN.cache_path(u) for u in REF + [U_C2, U_C0, U_RQ3]]
    W = t4.benchmark_workload(caches, n=60)
    SNAP = LlamaCppBackend(llm, t4.model_digest(PART_SHA), WHEEL_SHA256, snapshot=True)
    ab = t4.benchmark_ab(SNAP, W)
    t4.drop_snapshot_states(SNAP)
    ab["snapshot_canary_fingerprint"] = fingerprint(run_canary(
        lambda p: SNAP.generate(p["messages"], p["kind"], p["grammar"], p["max_tokens"]), SES.prompts))
    ab["fresh_canary_fingerprint"] = SES.fingerprint
    ab["ram_probe"] = t4.snapshot_ram_probe(SNAP, t4.wording_system_prompts(SES.prompts))
    t4.drop_snapshot_states(SNAP)
    S.save("bench_ab", ab)
if SESSION == 2:
    print({k: S.data["bench_ab"][k] for k in ("n", "seconds_A", "seconds_B", "saving_B", "B_repeatable", "A_equals_recorded")})

In [ ]:
# 12) SESSION 2: D53 benchmark C - two worker processes on real units (the kernel's model is closed first)
if SESSION == 2 and not S.done("bench_c"):
    SES.backend.llm.close(); llm = None; SES.backend.llm = None; gc.collect(); time.sleep(5)
    tele = t4.gpu_telemetry()
    assert int(float(tele.get("memory.used", "0"))) < 1500, f"GPU memory not freed: {tele}"
    B2 = [R.Unit("bench2", u.rq, u.split, u.kind, u.seed, u.condition, n_tasks=u.n_tasks) for u in (U_C2, U_C0)]
    if not (ROOT / "plans" / "bench2.json").exists():
        R.write_plan(ROOT, "bench2", B2, 1)
    parts_file = OUT / "parts.json"; parts_file.write_text(json.dumps(PART_SHA))
    start_at = time.time() + 480                                  # synchronised start after both loads and canaries
    procs = [subprocess.Popen([sys.executable, "-m", "eem_acc.stage3.llm.unit_worker", "--root", str(ROOT),
                               "--phase", "bench2", "--shard", f"{k}/2", "--model", str(MODEL_PATHS[0]),
                               "--parts", str(parts_file), "--wheel-sha", WHEEL_SHA256,
                               "--minilm-rev", PINS["minilm_revision"], "--session", "s33b-2c",
                               "--start-at", str(start_at), "--out", f"/content/w{k}.json"],
                              cwd=REPO_DIR, env=dict(os.environ, PYTHONPATH=f"{REPO_DIR}/src"),
                              stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True) for k in range(2)]
    peak = []
    while any(p.poll() is None for p in procs):
        peak.append(t4.gpu_telemetry()); time.sleep(30)
    errs = [p.communicate()[1][-2000:] for p in procs]
    outs = [json.loads(Path(f"/content/w{k}.json").read_text()) for k in range(2) if Path(f"/content/w{k}.json").exists()]
    base = sum(R.split_result(json.loads(RUN.result_path(u).read_text()))[1]["wall_seconds"] for u in (REF[0], U_C0))
    load = lambda u: R.split_result(json.loads(RUN.result_path(u).read_text()))[0]
    wall = (max(o["end"] for o in outs) - min(o["start"] for o in outs)) if len(outs) == 2 else None
    S.save("bench_c", {"workers": outs, "errors": errs, "wall_two_processes": wall, "wall_single_process": base,
                       "gain": (1 - wall / base) if wall else None,
                       "vram_peak_mib": max((int(float(t.get("memory.used", 0))) for t in peak), default=None),
                       "canary_equal_epoch0": all(o["canary_fingerprint"] == SES.fingerprint for o in outs),
                       "payload_vs_smoke": {u.unit_id: t4.compare_units(RUN.cache_path(u), RUN.cache_path(v), load(u), load(v))
                                            for u, v in zip(B2, (REF[0], U_C0))}})
if SESSION == 2:
    print({k: S.data["bench_c"][k] for k in ("wall_two_processes", "wall_single_process", "gain", "canary_equal_epoch0")})

In [ ]:
# 13) Done: stop the heartbeat, release the lease, download the results file - upload it to the chat
if SESSION == 2 and globals().get("SES") is not None:
    SES.close()
    from eem_acc.stage3.llm.ledger import summarise
    S.save("ledger", {k: v for k, v in summarise(ROOT).items() if k != "unit_wall_seconds"})
print("results file:", S.path, "| sections:", [k for k in S.data if k != "stamp"])
if IN_COLAB:
    from google.colab import files
    files.download(str(S.path))